In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import wikipedia as wp
import random

In [ ]:
def optimal_portfolio(df, sample_size = 10000, risk_free_rate = 0.02, min_weight = 0.05):

    returns = df.pct_change().dropna()
    trading_days = 252

    mean_returns = returns.mean()
    cov_matrix = returns.cov()
    results = np.zeros((3, sample_size))
    weights_list = []
    
    #Monte Carlo simulation
    for i in range(sample_size):
        weights = np.random.random(len(df.columns))
        weights = (weights / np.sum(weights)) * (1 - min_weight * len(df.columns)) + min_weight
        weights_list.append(weights)

        #computing portfolio volatility and expected returns to calculate the sharpe ratio
        port_return = (1 + np.sum(weights*mean_returns)) ** trading_days - 1    
        port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * trading_days, weights)))

        s_r = (port_return - risk_free_rate) / port_volatility 

        results[0,i] = port_return
        results[1,i] = port_volatility
        results[2,i] = s_r

    #locating the best portfolio from the simulation
    results_df = pd.DataFrame(results.T, columns = ["Expected Annual Return","Volatility","Sharpe Ratio"])
    max_sharpe_index = results_df["Sharpe Ratio"].idxmax()

    #retrieving portfolio information
    max_sharpe_portfolio = results_df.loc[max_sharpe_index]
    max_sharpe_weights = np.array(weights_list[max_sharpe_index])
    optimal_weights = dict(zip(returns.columns.tolist(), max_sharpe_weights))
    return_in_time = (returns.dot(max_sharpe_weights) + 1).cumprod()

    
    return max_sharpe_portfolio, optimal_weights

In [ ]:
#5-stock portfolio optimizer from a +5 stock sample (max sharpe)
def wallet(data, seed=123):
    random.seed(seed)
    data = data.dropna(axis = 1)
    R = data.pct_change().dropna()

    mean = R.mean()
    vol = R.std()
    score = mean/vol
    sorted = score.sort_values(ascending = False).head(50).index.to_list()

    best_sharpe = -5
    best_combo = None
    best_portfolio = None
    best_weights = None

    for i in range(100):
        combo = random.sample(sorted, 5)
        df = data[combo]
        port, weights = optimal_portfolio(df, sample_size=1000)

        if port["Sharpe Ratio"] > best_sharpe:
            best_sharpe = port["Sharpe Ratio"]
            best_combo = combo
            best_portfolio = port
            best_weights = weights

    print("Best 5-stock combo:", best_combo)
    print("Max Sharpe ratio:", best_sharpe)
    print("Weights:", best_weights)

In [5]:
#wikipedia S&P 500 scraper
page = wp.page("List of S&P 500 companies")
df = pd.read_html(page.html())[0]
list = df["Symbol"].str.replace(".", "-", regex=False).to_list()
#stock data download via yahoo
data = yf.download(list, start = "2020-01-01", end = "2025-01-01", auto_adjust = False)["Adj Close"]

C:\Users\szymo\AppData\Local\Temp\ipykernel_20244\411822130.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(page.html())[0]
[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['SNDK', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1577854800, endDate = 1735707600")')


In [11]:
wallet(data)

Best 5-stock combo: ['VST', 'LLY', 'KKR', 'WMT', 'TSLA']
Max Sharpe ratio: 2.218549470556576
Weights: {'VST': 0.2401080572031929, 'LLY': 0.4051473243380826, 'KKR': 0.05622546729976248, 'WMT': 0.0956663853510264, 'TSLA': 0.20285276580793565}
